In [ ]:
import os
import kagglehub

# Set your Kaggle API token as an environment variable
# Option 1: via terminal -> export KAGGLE_API_TOKEN="your_token_here"
# Option 2: place kaggle.json in ~/.kaggle/ directory
# os.environ["KAGGLE_API_TOKEN"] = "your_token_here"  # Do NOT hardcode your token here!

path = kagglehub.dataset_download("lipreadingankaya/clipped-word-dataset")
print("Dataset downloaded, path:", path)


In [ ]:
from torch.utils.data import Dataset, DataLoader
class LipReadingDataset(Dataset):
    def __init__(self, root_dir, subjects=None, max_frames=56, is_train=False):
        self.root_dir = root_dir
        self.max_frames = max_frames
        self.is_train = is_train
        self.samples = []

        ornek_kisi = os.listdir(root_dir)[0]
        ornek_yol = os.path.join(root_dir, ornek_kisi)
        self.classes = sorted([d for d in os.listdir(ornek_yol) if os.path.isdir(os.path.join(ornek_yol, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        all_subjects = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.current_subjects = subjects if subjects is not None else all_subjects

        for kisi in self.current_subjects:
            kisi_path = os.path.join(root_dir, kisi)
            for kelime in self.classes:
                kelime_path = os.path.join(kisi_path, kelime)
                if not os.path.exists(kelime_path): continue
                label = self.class_to_idx[kelime]
                for video_file in os.listdir(kelime_path):
                    if video_file.lower().endswith(('.mp4', '.mov')):
                        self.samples.append((os.path.join(kelime_path, video_file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        v_path, label = self.samples[idx]
        cap = cv2.VideoCapture(v_path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret: break
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if gray.shape != (96, 96):
                gray = cv2.resize(gray, (96, 96))
            frames.append(gray)
        cap.release()
        frames = np.array(frames, dtype=np.float32) / 255.0

        if self.is_train and len(frames) > 0:
            # 1. Horizontal flip
            if random.random() > 0.5:
                frames = frames[:, :, ::-1].copy()

            # 2. Brightness jitter
            frames = frames * random.uniform(0.8, 1.2)
            frames = np.clip(frames, 0, 1)

            # 3. Temporal jitter
            shift = random.randint(-3, 3)
            frames = np.roll(frames, shift, axis=0)

            # 4. Random erasing
            if random.random() > 0.5:
                t, h, w = frames.shape
                x = random.randint(15, 55)
                y = random.randint(15, 55)
                size = random.randint(10, 20)
                frames[:, y:y+size, x:x+size] = 0

            # 5. Gaussian noise
            noise = np.random.normal(0, 0.02, frames.shape).astype(np.float32)
            frames = np.clip(frames + noise, 0, 1)

        processed = np.zeros((self.max_frames, 96, 96), dtype=np.float32)
        real_len = min(len(frames), self.max_frames)
        if real_len > 0:
            processed[:real_len] = frames[:real_len]

        return (
            torch.from_numpy(processed).unsqueeze(0),
            torch.tensor(label),
            torch.tensor(real_len)
        )

In [ ]:
DATASET_PATH = "/root/.cache/kagglehub/datasets/lipreadingankaya/clipped-word-dataset/versions/1/dataset_lip"
dataset = LipReadingDataset(root_dir=DATASET_PATH)

# 1. all people
all_subjects = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])

# 2.  %85 train 15% test)
split_idx = int(0.85 * len(all_subjects))
train_subs = all_subjects[:split_idx]
val_subs = all_subjects[split_idx:]

train_ds = LipReadingDataset(root_dir=DATASET_PATH, subjects=train_subs, max_frames=56, is_train=True)
val_ds   = LipReadingDataset(root_dir=DATASET_PATH, subjects=val_subs,   max_frames=56, is_train=False)

# 3. DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Eğitimdeki Kişiler: {len(train_subs)} | Doğrulamadaki Kişiler: {len(val_subs)}")
print(f"Eğitim seti: {len(train_ds)} video | Doğrulama seti: {len(val_ds)} video")

In [ ]:
print(dataset.classes)

3D CNN + CONFORMER


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


# Conformer Block
class ConformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.3):
        super().__init__()

        # Self Attention
        self.attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)

        # Convolution Module
        self.conv_module = nn.Sequential(
            nn.BatchNorm1d(d_model),
            nn.Conv1d(d_model, d_model*2, 1),
            nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, 31, padding=15, groups=d_model),
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, 1),
            nn.Dropout(dropout)
        )

        self.norm2 = nn.LayerNorm(d_model)

        # Feed Forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)   # ← ekle
        )
        self.ff2 = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )

        self.norm4 = nn.LayerNorm(d_model)

        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):

        # 1. FF (½)
        x = x + 0.5 * self.ff(self.norm3(x))

        # 2. Attention
        normed = self.norm1(x)
        attn_out, _ = self.attn(
            normed,
            normed,
            normed,
            key_padding_mask=mask
        )
        x = x + attn_out

        # 3. Conv
        conv_out = self.conv_module(
            self.norm2(x).transpose(1, 2)
        ).transpose(1, 2)
        x = x + conv_out

        # 4. FF (½)
        x = x + 0.5 * self.ff2(self.norm4(x))

        return x

In [ ]:
class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=(1,2,2), padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(True),
            nn.Conv3d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm3d(out_ch),
        )
        self.downsample = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 1, stride=(1,2,2), bias=False),
            nn.BatchNorm3d(out_ch),
        )
        self.relu = nn.ReLU(True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.downsample(x))


class LipReadingConformer(nn.Module):
    def __init__(
        self,
        num_classes=55,
        d_model=512,
        nhead=8,
        num_layers=2
    ):
        super().__init__()

        # 3D CNN Frontend
        self.frontend = nn.Sequential(
            nn.Conv3d(
                1,
                64,
                kernel_size=(3, 5, 5),
                stride=(1, 2, 2),
                padding=(1, 2, 2),
                bias=False
            ),
            nn.BatchNorm3d(64),
            nn.ReLU(True),
            nn.MaxPool3d(
                kernel_size=(1, 3, 3),
                stride=(1, 2, 2),
                padding=(0, 1, 1)
            )
        )

        # ResNet Blocks (with skip connection)
        self.res1 = ResBlock3D(64, 128)
        self.res2 = ResBlock3D(128, 256)

        # Projection
        self.project = nn.Linear(256, d_model)

        self.pos_encoder = PositionalEncoding(d_model)

        # Conformer Layers
        self.conformer_layers = nn.ModuleList([
            ConformerBlock(d_model, nhead)
            for _ in range(num_layers)
        ])

        self.layer_norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, lengths):
        B, C, T, H, W = x.shape
        T_in = T
        device = x.device

        # 1. Frontend & ResNet
        x = self.frontend(x)
        x = self.res1(x)
        x = self.res2(x)  # (B, 256, T, H', W')

        # 2. Lengths scale
        T_out = x.shape[2]
        scale = T_out / T_in
        lengths_scaled = (lengths.float() * scale).long().clamp(min=1)

        # 3. Dual Pooling
        x_avg = F.adaptive_avg_pool3d(x, (None, 1, 1))
        x_max = F.adaptive_max_pool3d(x, (None, 1, 1))
        x = (x_avg + x_max).squeeze(-1).squeeze(-1)  # (B, 256, T)
        x = x.transpose(1, 2)                         # (B, T, 256)

        # 4. Project & Positional Encoding
        x = self.project(x)  # (B, T, d_model)
        x = self.pos_encoder(x)

        # 5. Maskeleme
        T_new = x.shape[1]
        mask = torch.arange(T_new).to(device).expand(B, T_new) >= lengths_scaled.unsqueeze(1)

        # 6. Conformer Layers
        for layer in self.conformer_layers:
            x = layer(x, mask=mask)

        # 7. Masked Temporal Mean
        mask_float = (~mask).float().unsqueeze(-1)  # (B, T, 1)
        x = (x * mask_float).sum(dim=1) / lengths_scaled.unsqueeze(1).float()

        # 8. LayerNorm + FC
        x = self.layer_norm(x)
        logits = self.fc(x)
        return logits

In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LipReadingConformer(num_classes=55).to(device)

dummy_vid = torch.zeros(1, 1, 10, 112, 112).to(device)
dummy_len = torch.tensor([10]).to(device)

with torch.no_grad():
    out = model(dummy_vid, dummy_len)
    print("Output shape:", out.shape)  # expected: [1, 55]

In [ ]:
# Mount Google Drive (if using Colab)
# from google.colab import drive
# drive.mount('/content/drive')
import os


In [ ]:
import cv2
import numpy as np
import random
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model ---
num_classes = len(dataset.classes)
model = LipReadingConformer(num_classes=num_classes).to(device)

# --- Loss, Optimizer, Scheduler ---
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6
)
scaler = torch.amp.GradScaler('cuda')

epochs = 50
print(f"Training on: {device} | Classes: {num_classes}")


In [ ]:
best_val_acc = 0.0

for epoch in range(epochs):

    # --- Train ---
    model.train()
    train_loss = 0

    for videos, labels, lengths in train_loader:
        videos, labels, lengths = videos.to(device), labels.to(device), lengths.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(videos, lengths)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    # --- Validation ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for videos, labels, lengths in val_loader:
            videos, labels, lengths = videos.to(device), labels.to(device), lengths.to(device)
            with torch.amp.autocast('cuda'):
                logits = model(videos, lengths)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total
    avg_train_loss = train_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']

    scheduler.step(val_acc)

    print(f"\n--- Epoch [{epoch+1}/{epochs}] ---")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Acc:    {val_acc:.2f}%")
    print(f"LR:         {current_lr:.8f}")
    print(f"Best Acc:   {best_val_acc:.2f}%")
    print("-" * 30)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc,
        'classes': dataset.classes,
    }, "last_checkpoint.pth")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"New best model saved! (Acc: {val_acc:.2f}%)")
        # Optional: copy to Drive
        # import shutil
        # shutil.copy("best_model.pth", "/content/drive/MyDrive/YOUR_FOLDER/best_model.pth")

print("Training complete. Best accuracy:", best_val_acc)


In [ ]:
# Resume training from best checkpoint
try:
    model.load_state_dict(torch.load("best_model.pth", map_location=device))
    print("Checkpoint loaded successfully. Resuming training...")
except FileNotFoundError:
    print("No checkpoint found. Starting from current model weights.")


In [ ]:
import gc

if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")


In [ ]:
# Set your checkpoint save directory
save_path = "./checkpoints"  # or "/content/drive/MyDrive/YOUR_PROJECT_FOLDER"
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory created: {save_path}")


In [ ]:
train_files = set([s[0] for s in train_ds.samples])
val_files = set([s[0] for s in val_ds.samples])

intersection = train_files.intersection(val_files)

def get_subject(path):
    return path.split('/')[-3]

train_subjects = set([get_subject(f) for f in train_files])
val_subjects = set([get_subject(f) for f in val_files])

subject_leak = train_subjects.intersection(val_subjects)

print(f"--- Yeni Sızıntı Raporu ---")
print(f"Aynı video hem train hem val'de var mı?: {'EVET! 🚨' if intersection else 'Hayır. ✅'}")
print(f"Train setindeki kişi sayısı: {len(train_subjects)}")
print(f"Val setindeki kişi sayısı: {len(val_subjects)}")
print(f"Aynı kişi hem train hem val'de var mı?: {'EVET! 🚨' if subject_leak else 'Hayır. ✅'}")

if subject_leak:
    print(f"⚠️ Sızan kişiler: {subject_leak}")
else:
    print("🚀 TEBRİKLER! Artık tamamen 'Subject-Independent' bir modelin var.")

## Evaluation

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report

MODEL_PATH = './checkpoints/best_model_2_layer.pth'  # Set your model path here

def load_model(model, path, device):
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Model loaded from: {path}")
    else:
        print(f"Model file not found: {path}")
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(model, MODEL_PATH, device)


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

all_preds = []
all_labels = []

print(f"Running evaluation on {len(val_ds)} unseen videos...")

model.eval()
with torch.no_grad():
    for videos, labels, lengths in tqdm(val_loader):
        videos, labels, lengths = videos.to(device), labels.to(device), lengths.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(videos, lengths)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy  = 100 * (np.array(all_preds) == np.array(all_labels)).sum() / len(all_labels)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print("-" * 30)
print(f"Accuracy:  {accuracy:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")
print("-" * 30)

cm = confusion_matrix(all_labels, all_preds)


In [ ]:
def plot_confusion_matrix(cm, class_names, normalize=False, figsize=(14, 12)):
    cm_to_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True) if normalize else cm.copy()
    fmt   = ".2f" if normalize else "d"
    title = "Normalized Confusion Matrix" if normalize else "Confusion Matrix"

    plt.figure(figsize=figsize)
    sns.heatmap(
        cm_to_plot,
        annot=len(class_names) <= 30,
        fmt=fmt,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

class_names = dataset.classes
plot_confusion_matrix(cm, class_names, normalize=False)


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

sorted_classes = sorted(
    [(k, v) for k, v in report.items() if isinstance(v, dict)],
    key=lambda x: x[1]['f1-score']
)
print("5 hardest classes:")
for label, metrics in sorted_classes[:5]:
    print(f"  {label}: F1={metrics['f1-score']:.2f}")


In [ ]:
import random

def run_batch_examples(model, dataset, device, class_names, num_examples=10):
    model.eval()
    print(f"{'TRUE WORD':<20} | {'PREDICTION':<20} | {'CONFIDENCE':<10} | STATUS")
    print("-" * 70)

    correct = 0
    with torch.no_grad():
        for _ in range(num_examples):
            idx = random.randint(0, len(dataset) - 1)
            video_tensor, label, length = dataset[idx]

            logits = model(video_tensor.unsqueeze(0).to(device), torch.tensor([length]).to(device))
            probs  = torch.nn.functional.softmax(logits, dim=1)
            conf, pred_idx = torch.max(probs, dim=1)

            true_word = class_names[label]
            pred_word = class_names[pred_idx.item()]
            status    = "✅" if true_word == pred_word else "❌"
            if true_word == pred_word:
                correct += 1

            print(f"{true_word:<20} | {pred_word:<20} | {conf.item()*100:>6.2f}%   | {status}")

    print("-" * 70)
    print(f"Accuracy on {num_examples} samples: {correct}/{num_examples} ({correct*10:.0f}%)")

run_batch_examples(model, val_ds, device, dataset.classes, num_examples=10)
